In [1]:
print("hello")

hello


In [16]:
from langchain.chat_models import init_chat_model

In [4]:
llm=init_chat_model(model='openai/gpt-oss-120b',model_provider='groq',temperature=0.5)

In [6]:
from pydantic import BaseModel, Field 

In [5]:
class Output(BaseModel):
    type:str=Field(description="""
                   * Type of the output. 
                   * Striclty standardize it to "QUERY" or "REMARK" """)
    value:str=Field(description="""
                    * If type is "QUERY" then provide valid Mysql query.
                    * It the type is "REMARK" then provide the short remark or feedback on the user query.
""")

In [6]:
structured_llm=llm.with_structured_output(Output)

In [76]:
try:
    TABLE_NAME
except NameError:
    TABLE_NAME = "st"  # Change this to the real table you want to query


def get_table_columns(table_name):
    try:
        cursor
    except NameError:
        return []
    try:
        cursor.execute(f"DESCRIBE {table_name}")
        return [f"    {row[0]}- {row[1]}" for row in cursor.fetchall()]
    except Exception:
        return []

columns = get_table_columns(TABLE_NAME)
if not columns:
    columns = [
        "    id- unique identifier of the student.",
        "    name- name of the student.",
        "    gender- gender of the student.",
        "    city- city where the student lives.",
        "    age- age of the student.",
        "    marks- marks of the student.",
        "    birthdate- birthdate of the student.",
    ]

columns_text = "\n".join(columns)

sytesm_prompt = f"""
- You are a expert mysql query generator.
- your task is to analyse the user request and if it is related to table schema defined below then generate valid mysql query without any special character or line seperators.
- If the user query is not related to the define schema then provide short remark.

* Table description:
* Table name: {TABLE_NAME}
* columns:
{columns_text}

user question is below:
"""

In [79]:
user_query="provide me city wise count from st table"
prompt=sytesm_prompt+user_query.lower().strip()

In [80]:
try:
    Output
except NameError:
    from pydantic import BaseModel, Field

    class Output(BaseModel):
        type: str = Field(description="""
                       * Type of the output. 
                       * Striclty standardize it to "QUERY" or "REMARK" """)
        value: str = Field(description="""
                        * If type is "QUERY" then provide valid Mysql query.
                        * It the type is "REMARK" then provide the short remark or feedback on the user query.
""")

try:
    sytesm_prompt
    user_query
    prompt
except NameError:
    sytesm_prompt = """
- You are a expert mysql query generator.
- your task is to analyse the user request and if it is related to table schema defined below then generate valid mysql query without any special character or line seperators.
- If the user query is not related to the define schema then provide short remark.

* Table description:
* Table name: students
* columns: 
    id- unique identifier of the student.
    name- name of the student.
    gender-gender of the student.
    city- city where the student lives.
    age- age of the student.
    marks-marks of the student.
    birthdate- birthdate of the student.

user question is below:
"""
    user_query = "provide me city wise count of students"
    prompt = sytesm_prompt + user_query.lower().strip()

try:
    structured_llm
except NameError:
    from langchain.chat_models import init_chat_model

    llm = init_chat_model(model='openai/gpt-oss-120b', model_provider='groq', temperature=0.5)
    structured_llm = llm.with_structured_output(Output)

response = structured_llm.invoke(prompt)

In [10]:
response

Output(type='QUERY', value='SELECT city, COUNT(*) AS student_count FROM students GROUP BY city')

In [35]:
import os
from dotenv import load_dotenv
import mysql.connector

load_dotenv()

DB_HOST = os.getenv("MYSQL_HOST", "localhost")
DB_PORT = os.getenv("MYSQL_PORT", "3306")
DB_USER = os.getenv("MYSQL_USER")
DB_PASSWORD = os.getenv("MYSQL_PASSWORD")
DB_NAME = os.getenv("MYSQL_DATABASE")

if not DB_USER or not DB_PASSWORD or not DB_NAME:
    raise ValueError(
        "Missing MySQL credentials in .env or environment variables."
    )

connection = mysql.connector.connect(
    host=DB_HOST,
    port=int(DB_PORT),
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)

In [28]:
cursor = connection.cursor()

In [58]:
# Use one of your actual database tables here.
TABLE_NAME = "st"  # Your table with data

print("Available tables:")
try:
    cursor.execute("SHOW TABLES")
    print([row[0] for row in cursor.fetchall()])
except Exception as exc:
    print("Could not list tables:", exc)

print(f"Using TABLE_NAME = {TABLE_NAME}")
try:
    cursor.execute(f"DESCRIBE {TABLE_NAME}")
    for row in cursor.fetchall():
        print(row)
except Exception as exc:
    print("Could not describe table", TABLE_NAME, exc)

Available tables:
['city_mumbai', 'hobby', 'st', 'st_bkc', 'students']
Using TABLE_NAME = st
('id', 'int', 'NO', 'PRI', None, '')
('name', 'varchar(25)', 'NO', '', None, '')
('gender', 'varchar(2)', 'NO', '', None, '')
('city', 'varchar(25)', 'YES', '', 'Mumbai', '')
('age', 'int', 'NO', '', None, '')
('marks', 'float', 'NO', '', None, '')
('birthdate', 'date', 'YES', '', None, '')


In [82]:
cursor.execute(response.value)
rows=cursor.fetchall()

In [39]:
rows

[]

In [53]:
output_system_prompt="""
* You are a expert data presenter.
* You will be provided with result of a mysql query.
* Present that data in a user friendly format using correct visualization.
- dont suggest any additional things. just provide the visualization or representation of the data.

* the mysql query result is provided below:
"""

In [83]:
output_prompt=output_system_prompt+str(rows)

In [84]:
output_response=llm.invoke(output_prompt)

In [85]:
print(output_response.content)

**City – Count (Horizontal Bar Chart)**  

```
Mumbai      █████ 5
Delhi       ███   3
Pune        ███   3
Chennai     ███   3
Hyderabad   ███   3
Kolkata     ██    2
Kochi       ██    2
Agra        ██    2
Jaipur      ██    2
Surat       ██    2
Lucknow     ██    2
Ahmedabad   █     1
Nagpur      █     1
```
